# 2. Time-Series Decomposition via Codifferencing


# Time Series Basics

A **time series** is a sequence of observations recorded in time order. Examples include daily COVID-19 cases, hourly electricity demand, monthly rainfall, and daily stock prices.

### Analogy: a patient monitor
A hospital monitor records heart rate repeatedly. The readings are not just a bag of numbers: their order matters. A sudden jump after a stable period can be meaningful because it happened at a particular time. Time-series analysis works with this temporal structure.

## Terms used in this project

| Term | Meaning | Analogy |
|---|---|---|
| Observation | A measured value at a particular time | One monitor reading |
| Time index | The date/time attached to an observation | Timestamp on the reading |
| Trend | Long-term direction | A road gradually going uphill |
| Seasonality | A repeating pattern with a known period | Weekday traffic every morning |
| Cycle | Repeated oscillation that need not have a perfectly fixed period | Waves rising and falling |
| Lag | Distance back in time used for comparison | Looking 7 days into the past |
| Residual | Signal left after modeled structure is removed | Noise left after removing a melody |
| Stationarity | Statistical behavior that is reasonably stable over time | A machine staying around its normal range |
| Autocorrelation | Relationship between observations at different lags | Today partly remembering yesterday |
| Periodicity | Repetition over an interval | A weekly alarm |
| Heavy tail | Relatively frequent extreme observations | Very large vehicles appearing more often than expected |
| Forecast | Prediction of future observations | Estimating tomorrow's traffic |

## Why time order matters

Rows in ordinary tabular data can often be shuffled. Shuffling a time series destroys temporal relationships. For COVID-19 data, moving a rise in cases to a random date would remove the timing information we want to study.

## Project analogy: a song

Think of the observed COVID-19 series as a **song**: the observed series is the complete song, trend is the broad musical direction, seasonality/cycles are repeating rhythm, codifferencing is a transformation used to remove important structure, the residual is what remains, stationarity checks ask whether the remaining signal behaves consistently, Hill estimation examines extreme values, PAR models use previous observations at relevant positions, and MAE/RMSE measure prediction error. This analogy is only for intuition; the notebook computations are the actual analysis.


## Objective

Prepare the India COVID-19 daily new-cases series and apply the codifference-based transformation used in the original analysis.

- **Trend lag = 14:** a 14-day lag is used in the trend-removal stage.
- **Seasonal lag = 7:** a 7-day lag captures weekly structure in daily observations.
- **Residual series:** the transformed series passed to later diagnostics and PAR models.

### Analogy
Imagine a daily temperature record with a slow change plus a repeating weekly effect. We transform the series so the large-scale structure is reduced and the remaining signal can be modeled more directly.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. Load the dataset
df = pd.read_csv("owid-covid-data.csv")
df_india = df[df['location'] == 'India'][['date', 'new_cases']].copy()

# 2. Preprocess the data
df_india['date'] = pd.to_datetime(df_india['date'])
df_india.set_index('date', inplace=True)
df_india = df_india.asfreq('D')  # Ensure daily frequency
df_india['new_cases'] = df_india['new_cases'].fillna(0)  # Avoid chained assignment
series = df_india['new_cases']

# 3. Codifference function (robust for both Series and NumPy array input)
def codifference(x, lag=1):
    x = np.asarray(x)
    return x[lag:] - x[:-lag]

# 4. Apply codifference-based detrending
trend_lag = 14  # Lag for local trend removal
detrended = codifference(series, lag=trend_lag)
detrended_index = series.index[trend_lag:]

# 5. Apply codifference-based deseasonalizing
seasonal_lag = 7  # Lag for weekly seasonality
deseasonalized = codifference(detrended, lag=seasonal_lag)
final_index = detrended_index[seasonal_lag:]

# ======================
# 6. PLOTS
# ======================

# Plot original series
plt.figure(figsize=(14, 4))
plt.plot(series, label='Original New Cases')
plt.title('Original Daily New COVID-19 Cases (India)')
plt.xlabel('Date')
plt.ylabel('New Cases')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot codifference-based detrended series
plt.figure(figsize=(14, 4))
plt.plot(detrended_index, detrended, label=f'Detrended (lag={trend_lag})', color='orange')
plt.title('Codifference-Based Detrended Series')
plt.xlabel('Date')
plt.ylabel('Detrended Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot final residual after deseasonalization
plt.figure(figsize=(14, 4))
plt.plot(final_index, deseasonalized, label=f'Deseasonalized Residual (lag={seasonal_lag})', color='purple')
plt.title('Final Residual (Codifference-Based Detrended + Deseasonalized)')
plt.xlabel('Date')
plt.ylabel('Residual Value')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Save the derived residual series


In [ ]:
import os
os.makedirs("data", exist_ok=True)
residual_series.to_csv("data/residual_series.csv", header=True, index_label="date")
print("Saved residual series to data/residual_series.csv")
